# Fase 8 — Fine-tuning neural no Colab (XLM-R · BERTimbau · BERTweet)

**Turnkey.** Antes de tudo, gere o pacote localmente:
```
.venv/Scripts/python.exe notebooks/make_colab_bundle.py
```
e suba `colab_bundle.zip` para o seu Google Drive.

Depois, aqui no Colab: **Runtime → Change runtime type → GPU (T4)** e rode as células em ordem.

Os transformers usam o MESMO corpus congelado (UTF-8 corrigido), o MESMO ajuste de limiar
na validação e o MESMO schema de métricas/predições dos modelos clássicos — então entram
direto no leaderboard, McNemar e calibração locais, sem ajuste. A comparação clássico vs
neural fica justa por construção.


## 1 · GPU + Google Drive


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')


## 2 · Desempacotar o bundle
Ajuste `BUNDLE` se você subiu o zip para outra pasta do Drive.


In [ ]:
import os, zipfile, pathlib
BUNDLE = '/content/drive/MyDrive/colab_bundle.zip'   # <-- ajuste se preciso
DEST = '/content/hsc'
assert os.path.exists(BUNDLE), f'nao achei {BUNDLE} — suba colab_bundle.zip ao Drive'
pathlib.Path(DEST).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z: z.extractall(DEST)
%cd /content/hsc
print(sorted(os.listdir('.')))


## 3 · Dependências (a GPU stack do HF por cima do torch do Colab)


In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps


## 4 · Sanidade + trava anti-mojibake
Confere que o corpus tem `split` e que o PT está em UTF-8 correto (não latin-1).


In [ ]:
import pandas as pd
from hsc.config import resolve
c = pd.read_parquet(resolve('data/processed/corpus_strict.parquet'))
assert 'split' in c.columns, 'corpus sem split — rode hsc split antes de empacotar'
pt = ' '.join(c[c.source_dataset=='pt_fortuna'].text_clean.head(2000))
assert ('\u00e9' in pt) or ('\u00e3' in pt), 'PT sem acentos reais — regere o bundle'
assert '\u00c3\u00a9' not in pt, 'mojibake latin-1 (Ã©) detectado no PT!'
print('corpus OK:', c.split.value_counts().to_dict())


## 5 · Treinar

Matriz: 3 configs × 2 políticas × N seeds. Comece com **1 seed** para ter a comparação;
depois troque `SEEDS = [42, 43, 44]` para média ± desvio (3× o tempo).

O laço é **resumível**: se a sessão cair, rode a célula de novo e ele pula o que já terminou.
Estimativa T4: XLM-R ~15-25 min/run; BERTimbau/BERTweet um pouco menos (filtram por idioma).


In [ ]:
import os, yaml
from hsc.config import resolve
from hsc.train_neural import train_neural_from_config

CONFIGS  = ['configs/neural/xlmr_multilingual.yaml',
            'configs/neural/bertimbau_pt.yaml',
            'configs/neural/bertweet_en.yaml']
POLICIES = ['strict', 'broad']
SEEDS    = [42]            # -> [42, 43, 44] para IC no artigo

def _name(cfg): return yaml.safe_load(open(cfg))['name']
def _done(cfg, pol, sd):
    return os.path.exists(resolve(f'reports/metrics/{_name(cfg)}_{pol}_s{sd}.json'))

for cfg in CONFIGS:
    for pol in POLICIES:
        for sd in SEEDS:
            if _done(cfg, pol, sd):
                print('skip (ja feito):', _name(cfg), pol, sd); continue
            print('==> treinando', _name(cfg), pol, 'seed', sd, flush=True)
            train_neural_from_config(cfg, policy=pol, seed=sd)
print('treino concluido')


## 6 · Empacotar resultados de volta para o Drive
Leva só métricas + predições + registry (os pesos ficam no Colab; regenere se precisar).


In [ ]:
import glob, zipfile
from hsc.config import resolve
out = '/content/drive/MyDrive/hsc_neural_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(str(resolve('reports/metrics'))+'/*.json'):
        z.write(f, 'reports/metrics/'+os.path.basename(f))
    for f in glob.glob(str(resolve('reports/predictions'))+'/*.parquet'):
        z.write(f, 'reports/predictions/'+os.path.basename(f))
    reg = resolve('models/registry.json')
    if os.path.exists(reg): z.write(reg, 'models/registry_neural.json')
print('resultados salvos em', out)


## 7 · De volta ao local (fecha a comparação)

Baixe `hsc_neural_results.zip` do Drive para a raiz do repositório e rode:
```
.venv/Scripts/python.exe notebooks/merge_neural_results.py hsc_neural_results.zip
hsc report      # leaderboard clássico + neural juntos
hsc analyze     # McNemar + calibração incluindo os neurais
hsc bias        # viés de identidade dos neurais também
```
O merge copia métricas/predições e funde as entradas neurais no registry local
(sem tocar nas clássicas).
